# Lab 04 · Tool calling: read and change an Order

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## Let an agent read and change an Order safely
**Start with a problem:** A model asked for an order total cannot see the current order just by guessing, and a free-form write request may be unsafe.

**In this lab:** Expose a read tool and a checked write tool. The Order root still controls its own state.

**So far:** Lab 03 proposed ShopEasy's business boundaries.

**New:** tool calling. A model requests a named function with arguments; our code validates the request and runs it. The functions belong to the Order aggregate root, which controls changes and maintains the total.

The invariant is **total = sum of line quantities × unit prices**. Outside callers cannot set the total through the tool interface. They request `add_item`, and Order keeps its own invariant true. All state is an in-memory simulation; no actual order or payment is created.

Order is an entity with a tracked identity. Money is a value object compared by amount and currency. Here integer cents and line dictionaries deliberately simplify a fuller OrderLineItem and Money model; the root still protects the total invariant.

**At a glance**

![Lab 04 at a glance: Request, then Pick a tool ({"tool", "args"}), then call_tool() (name · types), then User confirms (write tools only), then read_order() (snapshot only) or add_item() (root protects total), then Answer (from the result).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/04.svg)

## Think about the example

An Order has a stable identity and contains order lines. Its root is responsible for keeping the total equal to the sum of its lines. Money is compared by amount and currency.

**Concepts:** Aggregate root; Entity vs value object; Consistency boundary and invariant.

Use the scenario above to consider these questions before running the code:

1. Why must outside code change an Order through the aggregate root rather than set its total?
2. What distinguishes the identity of Order from the value of Money, and which details does this tiny model omit?

## The idea in context

Tool calling connects model-selected requests to ordinary code. The host validates requests and retains authority over state changes.

Tools perform actions; the model cannot bypass the aggregate root to edit totals.

| Key term | Meaning |
| --- | --- |
| **Tool** | A function that the model may ask the program to run. |
| **Function calling** | A reply format in which the model names a tool and gives its arguments as JSON. |
| **Read and write tools** | A read tool only looks data up; a write tool changes data and needs more care. |
| **Confirmation** | A person's approval before a tool changes anything. |

## Pattern · One-Step Tool Agent

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Incremental Tool Agent, Registry & Adapter.*

| | |
| --- | --- |
| **Context** | Many tasks need a deterministic capability, such as a lookup, before an answer can be trusted. |
| **Problem** | Free-form model output is hard to execute, while a long reasoning chain is unnecessary for a straightforward tool call. |
| **Solution** | Ask the model for its plan, a tool name, and parameters as JSON; execute the tool; then use the result. |
| **Use when** | One tool is enough per step; latency matters; tool schemas are simple and stable. |
| **Trade-off** | Fast and compact, but tool choice and parameters are made in one shot. The Incremental Tool Agent separates them, at a higher cost. |
| **Flow** | Goal → Plan + choose → Tool → Synthesize |
| **Quality attributes** | Benefits: Latency, Cost, Performance, Maintainability. Liabilities: Reliability, Accuracy, Security, Observability. |
| **In this lab** | The model chooses `read_order` or `add_item`. The dispatcher validates JSON, names and argument types; the aggregate root preserves its total invariant. Writes in this simulation require confirmation. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 12 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.0" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/fc798d4d9fda69756f7d860756fe7c0507f7f2a2.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Step 1 · Put the invariant behind the aggregate root
Prices use integer cents. A snapshot copies the lines so the caller does not receive mutable internal state. Python's private naming is a teaching convention, not a security boundary. A class groups state and the operations that may change it. `self` is the current Order; `__init__` creates it. `dict(line)` copies each line so a caller cannot edit the stored dictionaries through a returned snapshot.

In [ ]:
import json

class Order:
    def __init__(self, order_id):
        self.order_id = order_id
        self.__lines = []
        self.__total = 0

    def read(self):
        copied_lines = [dict(line) for line in self.__lines]
        return {"order_id": self.order_id, "lines": copied_lines, "total_cents": self.__total}

    def add_item(self, product: str, quantity: int, price_cents: int):
        if not product.strip() or quantity < 1 or price_cents < 0:
            return {"error": "product required, quantity >= 1, price_cents >= 0"}
        line = {"product": product, "quantity": quantity, "price_cents": price_cents}
        self.__lines.append(line)
        total = 0
        for stored_line in self.__lines:
            total += stored_line["quantity"] * stored_line["price_cents"]
        self.__total = total
        return self.read()

order = Order("order-17")
print(order.add_item("Jacket", 1, 4200))

## Step 2 · Register a read tool and a write tool
The aggregate decides what changes are legal. The tool registry exposes only its public operations. In a real service these operations would also need authorization, concurrency control and a local transaction. Each registry entry stores the callable under `run`, its argument names/types under `args`, and whether it writes. For example, `order.read` stores the method; the final `()` calls it.

In [ ]:
TOOLS = {
    "read_order": {
        "run": order.read, "args": {}, "writes": False,
        "description": "Read the current Order snapshot.",
    },
    "add_item": {
        "run": order.add_item,
        "args": {"product": str, "quantity": int, "price_cents": int},
        "writes": True,
        "description": "Add an Order line through its root; total is recomputed in cents.",
    },
}
print(TOOLS["read_order"]["run"]())

## Step 3 · Validate the model's request
A request must be an object with a known tool and exactly the declared argument names/types. Even valid JSON such as `[]` is rejected. The write requires a confirmation flag before entering the aggregate root. Step 5 sets this flag in code to simulate approval; it does not ask a person. `**arguments` passes the checked dictionary as named function arguments. We use `type(value) is ...` so `True` cannot count as an integer quantity.

In [ ]:
def call_tool(reply, confirmed=False):
    try:
        request = json.loads(reply)
        if not isinstance(request, dict):
            return {"error": "request must be an object"}
        spec = TOOLS.get(request.get("tool"))
        if spec is None:
            return {"error": "unknown tool"}
        arguments = request.get("args", {})
        if not isinstance(arguments, dict) or set(arguments) != set(spec["args"]):
            return {"error": "argument names do not match the schema"}
        for name, value in arguments.items():
            if type(value) is not spec["args"][name]:
                return {"error": "argument types do not match the schema"}
        if spec["writes"] and not confirmed:
            return {"error": "confirmation required before changing the Order"}
        return spec["run"](**arguments)
    except (json.JSONDecodeError, TypeError) as error:
        return {"error": str(error)}

print(call_tool('[]'))

## Step 4 · Let the model select a read tool
The model receives the tool descriptions and schemas, not private state. It must read the actual total instead of guessing one. `describe()` creates plain text for the model from the registry. `python_type.__name__` turns a Python type such as `int` into the word `int`.

In [ ]:
def describe():
    descriptions = []
    for name, spec in TOOLS.items():
        argument_types = {}
        for argument, python_type in spec["args"].items():
            argument_types[argument] = python_type.__name__
        descriptions.append(f"{name}: {spec['description']} args: {argument_types}")
    return "\n".join(descriptions)

FORMAT = 'Reply with JSON only, no code fences: {"tool":"name","args":{...}}'
prompt = f"Tools:\n{describe()}\nRequest: What is Order order-17's current total?\n{FORMAT}"
choice = ask(prompt)
result = call_tool(choice)
prompt = f"Order tool result: {json.dumps(result)}. State the total in cents using only this result."
answer = ask(prompt)
print("Requested:", choice)
print("Tool result:", result)
print("Model answer:", answer)

## Step 5 · Change the Order through its root
The model asks to add two scarves, each 1500 cents. Confirmation authorizes the simulation's write. The total must become 7200 cents; attempting a direct `set_total` fails because that tool does not exist. Run this write cell once. To repeat the whole example, rerun Steps 1–5 so the Order starts fresh; repeating only Step 5 would add the scarves again.

In [ ]:
prompt = f"Tools:\n{describe()}\nRequest: Add 2 scarves at 1500 cents each to order-17.\n{FORMAT}"
choice = ask(prompt)
print("Before confirmation:", call_tool(choice))
print("After confirmation:", call_tool(choice, confirmed=True))
print("Direct setter:", call_tool('{"tool":"set_total","args":{"total_cents":0}}', confirmed=True))
snapshot = order.read()
line_sum = sum(line["quantity"] * line["price_cents"] for line in snapshot["lines"])
assert snapshot["total_cents"] == line_sum
assert snapshot["total_cents"] == 7200, "The requested item change was not completed correctly."

## What you should see
The order starts with one jacket (4200 cents). The read tool exposes that value. The write is refused before confirmation, then adds two scarves and updates the total to 7200. A direct total setter is rejected. The invariant is checked independently.

## Try it
1. After resetting with Step 1 and rerunning Steps 2–3, call `call_tool` with a negative quantity and `confirmed=True`. Compare `order.read()` before and after.
2. Call `call_tool` with a JSON request containing `"quantity":"2"`. The dispatcher should report a type error. Which boundary rejected it?
3. Explain why a model must not update Order and Inventory tables in one cross-service transaction. Relate your answer to small aggregates and sagas.
4. Change the returned snapshot's first line. Does the stored Order change?

## Takeaway
Tools give a model capabilities; contracts and aggregate-root methods keep business rules in deterministic code. The agent never receives a tool for bypassing the invariant.